# SMS Spam Detection Pipeline
This notebook demonstrates an end-to-end Machine Learning pipeline to classify SMS messages as either **Ham** (legitimate) or **Spam**.

### Workflow:
1. **Data Loading & Exploration:** Initial profiling of the raw data.
2. **Data Preprocessing:** Handling duplicate entries and encoding categorical target labels.
3. **Data Splitting:** Partitioning into training and evaluation sets.
4. **Feature Extraction:** Vectorizing textual data into numerical features using the Bag-of-Words model (`CountVectorizer`).
5. **Model Training & Evaluation:** Training and validating Naive Bayes and Logistic Regression classifiers.
6. **Inference:** Evaluating custom, unseen inputs.

In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import all required libraries
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

## 1. Data Loading and Exploration
We begin by reading the tab-separated data file and computing descriptive metrics to understand class distributions, duplicate entries, and missing values.

In [3]:
# Load the tab-separated SMS Spam Collection dataset
df = pd.read_csv("SMSSpamCollection", sep="\t", names=["label", "message"])

print("--- First 5 Records ---")
display(df.head())

print(f"\nDataset Shape: {df.shape}")

print("\n--- Dataset Profile Info ---")
df.info()

print("\n--- Label Distribution Counts ---")
print(df["label"].value_counts())

print("\n--- Label Distribution Proportion ---")
print(df["label"].value_counts(normalize=True))

print(f"\nMissing Values per Column:\n{df.isnull().sum()}")
print(f"\nTotal Duplicate Rows: {df.duplicated().sum()}")

--- First 5 Records ---


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



Dataset Shape: (5572, 2)

--- Dataset Profile Info ---
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB

--- Label Distribution Counts ---
label
ham     4825
spam     747
Name: count, dtype: int64

--- Label Distribution Proportion ---
label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

Missing Values per Column:
label      0
message    0
dtype: int64

Total Duplicate Rows: 403


## 2. Data Preprocessing & Cleaning
In this phase, duplicate entries are removed to prevent overfitting or validation bias. Categorical targets (`ham`, `spam`) are converted to clean numerical flags (`0`, `1`).

In [4]:
# Remove duplicate entries in-place
df.drop_duplicates(inplace=True)
print(f"Dataset Shape after Removing Duplicates: {df.shape}")

# Map textual categories to numerical labels: 'ham' -> 0, 'spam' -> 1
df["label"] = df["label"].map({"ham": 0, "spam": 1})

print("\n--- Processed DataFrame Structure ---")
display(df.head())

Dataset Shape after Removing Duplicates: (5169, 2)

--- Processed DataFrame Structure ---


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


## 3. Train-Test Splitting
We break the data into independent features ($X$) and targets ($y$). We perform an 80/20 train/test split, locking down seed randomness via `random_state`.

In [5]:
# Separate features (X) and target variable (y)
X = df["message"]  # SMS text strings
y = df["label"]  # Encoded labels

# Split into 80% Training and 20% Testing subsets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training split size: {X_train.shape[0]}")
print(f"Testing split size : {X_test.shape[0]}")

Training split size: 4135
Testing split size : 1034


## 4. Text Vectorization (Feature Extraction)
Computers process matrices better than raw text. We instantiate a `CountVectorizer` to generate a token-vocabulary map. 

* **`fit_transform`** on training sets creates vocabulary matrices.
* **`transform`** on test matrices standardizes feature counts back onto that same vocabulary index.

In [6]:
# Initialize Bag of Words transformer
vectorizer = CountVectorizer()

# Build vocabulary dictionary and transform the raw text training data
X_train_vector = vectorizer.fit_transform(X_train)

# Map testing raw text onto existing vocabulary space
X_test_vector = vectorizer.transform(X_test)

## 5. Model Selection, Training & Evaluation
We build a comparison dictionary containing both a standard Text-Classification classic (**Multinomial Naive Bayes**) and a linear classifier (**Logistic Regression**).

In [7]:
# Initialize the target models
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(),
}

# Evaluate performance metrics across both classifiers
for name, model in models.items():
    print(f"\n==================== {name.upper()} ====================")

    # Fit the predictive model
    model.fit(X_train_vector, y_train)

    # Perform evaluation on the test slice
    y_pred = model.predict(X_test_vector)

    # Calculate and show final scores
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {accuracy:.4f}\n")

    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


==================== MULTINOMIAL NAIVE BAYES ====================
Accuracy: 0.9816

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       894
           1       0.94      0.92      0.93       140

    accuracy                           0.98      1034
   macro avg       0.96      0.96      0.96      1034
weighted avg       0.98      0.98      0.98      1034

Confusion Matrix:
[[886   8]
 [ 11 129]]

==================== LOGISTIC REGRESSION ====================
Accuracy: 0.9778

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       894
           1       0.98      0.86      0.91       140

    accuracy                           0.98      1034
   macro avg       0.98      0.93      0.95      1034
weighted avg       0.98      0.98      0.98      1034

Confusion Matrix:
[[891   3]
 [ 20 120]]


## 6. Custom Unseen Inference
Below, we supply a raw sample string array simulating an upcoming schedule confirmation message to test the finalized inference path.

In [8]:
# Sample mock input
email = ["Meeting tomorrow at 11 am."]

# Convert text into matching vocabulary space
email_vector = vectorizer.transform(email)

# Use the last evaluated model instance to test prediction mapping
prediction = model.predict(email_vector)
probability = model.predict_proba(email_vector)

# Print metrics outputs
print(f"Raw Prediction Category: {prediction[0]}")
print(f"Classification Label   : {'Spam' if prediction[0] == 1 else 'Ham'}")
print(f"Spam Probability       : {probability[0][1]:.4f}")
print(f"Ham Probability        : {probability[0][0]:.4f}")

Raw Prediction Category: 0
Classification Label   : Ham
Spam Probability       : 0.0027
Ham Probability        : 0.9973
